# Intro
In this notebook, the results of querying the segmentation counts are visualized. This is done in interactive plots (using Dash).

In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
from natsort import natsorted

import dash
from dash import dcc, html, Input, Output
import plotly.express as px

import cv2

In [ ]:

qdf = pd.read_csv('queries_df.csv')
qdf.set_index('frame', inplace=True)
qdf.sort_index(inplace=True)
qdf = qdf.loc[natsorted(qdf.index)]
qdf.head()

In [ ]:
# Plot results as heatmap

plt.figure(figsize = (18,5))
ax = sns.heatmap(qdf.T)
plt.title('Count per query for each frame')
plt.xlabel('Frame')

# Force integer values on the colorbar
colorbar = ax.collections[0].colorbar  # Get colorbar
colorbar.locator = ticker.MaxNLocator(integer=True)  # Set integer ticks
colorbar.update_ticks()  # Apply the new ticks

plt.show()

# Interactive plots


Using Dash: zoomable, show selected frames, and store them in a list to view the frames later on. 

In [ ]:
# Global variable to store selected frames
selected_frames = []

x_labels = list(qdf.T.columns)

# Example heatmap
fig = px.imshow(
    qdf.T,
    aspect="auto",
    color_continuous_scale="magma"
)
fig.update_layout(yaxis=dict(fixedrange=True))  # lock y-axis

# Create Dash app
app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Graph(id="heatmap", figure=fig),
    html.Div(id="output", style={"color": "grey"})  # to display selected frames
])

# Callback to capture zoom/selection on x-axis
@app.callback(
    Output("output", "children"),
    Input("heatmap", "relayoutData")
)
def capture_xrange(relayout_data):
    global selected_frames  
    if relayout_data and "xaxis.range[0]" in relayout_data:
        x_start = relayout_data["xaxis.range[0]"]
        x_end = relayout_data["xaxis.range[1]"]

        start_idx = max(0, int(round(x_start)))
        end_idx = min(len(x_labels) - 1, int(round(x_end)))

        selected_frames = x_labels[start_idx:end_idx + 1]
        return f"Selected frames: {selected_frames}"
    
    selected_frames = []
    return "Select an x-range by zooming or dragging."


app.run(mode="inline")




# Interactive image viewer
Display the selected frames. 

In [10]:
folder = '/path/to/folder_with_frames/'

In [ ]:
idx = 0           # current index

while True:
    img = cv2.imread(folder + selected_frames[idx])

    fname = selected_frames[idx]
    frame_text = f"{fname} [{idx}/{len(selected_frames)-1}]"

    # Overlay filename and index
    cv2.putText(
        img,
        frame_text,
        (30, 30),                # position (x, y)
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,                     # font scale
        (0, 255, 0),             # green text
        2,                       # thickness
        cv2.LINE_AA
    )

    # Show window with dynamic title: filename + index
    cv2.imshow("Frame Viewer", img)

    key = cv2.waitKey(0)


    if key == ord("q"):  # quit
        break
    elif key == 83:  # right arrow
        idx = (idx + 1) % len(selected_frames)
    elif key == 81:  # left arrow
        idx = (idx - 1) % len(selected_frames)

cv2.destroyAllWindows()
